In [ ]:
from pathlib import Path
import markdown_it
import re
import sys
from scipy import stats
from pandas import DataFrame as df
from typing import Generator, List
import json
from content_vacuum import ace_language

In [ ]:
# clean up some ace tags
f = "tool_use_guidelines.md"
claude_memory = "memory.json"
# some utilities
"""fix the bad tags and write the results back to the file"""
def tag_cleanup(cleanup_file):
    tagRegex = re.compile(r'@@ACE:(?P<tag>\[[A-z]+-[0-9]+\])@@')
    content = ''
    with open(cleanup_file, 'w+', encoding='utf-8') as fd:
        content = fd.read()
        while results := tagRegex.search(content):
            content = content.replace(results.group(), results.groups()[0])
        fd.seek(0)
        fd.write(content)

def parse_jsonl_file(filename:str) -> Generator[dict, None, None]:
    """parse a jsonl file, return a generator of dicts for each entry of the jsonl file
    Note: If I was to accumulate the list, this would be O(n), but the generator method is O(1)
    """
    with open(filename, 'r', encoding='utf-8') as fd:
        lines = fd.readlines()
        for line in lines:
            try:
                # yield the single entry
                yield json.loads(line)
            except json.decoder.JSONDecodeError as e:
                print("invalid line: ", lines.index(line), "error was: ", e.msg)
                sys.exit(2)

def prune_ace_entries_from_object(entries:Generator[dict, None, None]) -> Generator[dict, None, None]:
    """Remove ace entries from the observations list for each entry"""
    for entry in entries:
        if 'observations' in entry:
            entry['observations'] = [obs for obs in filter(lambda e: e.find('@@ACE:') == -1, entry['observations'])]
        yield entry



def write_ace_memory_entries(entries:Generator[dict, None, None], outputfile:str) -> None:
    with open(outputfile, 'w+', encoding='utf-8') as fd:
        for entry in entries:
            if 'observations' in entry:
                for observation in entry['observations']:
                    if observation.find('@@ACE') == 0:
                        fd.write(observation)
                        fd.write('\n\n')

def extract_ace_memory_entries_to(inputfile:str, outputfile:str) -> None:
    write_ace_memory_entries(parse_jsonl_file(inputfile), outputfile)


def extract_claude_memory_to_ace_entries():
    claude_memory = "memory.json""
    output_entry_file = "MEMORY_TAG_PLAYBOOK.md"
    extract_ace_memory_entries_to(claude_memory, output_entry_file)

def get_claude_mem_without_ace():
    claude_memory = "memory.json"
    elements = parse_jsonl_file(claude_memory)
    return prune_ace_entries_from_object(elements)


In [ ]:
#extract_claude_memory_to_ace_entries()
items = get_claude_mem_without_ace()
l = [x for x in items if 'observations' in x and x['name'] == 'default_user']
json.dumps(l[0])

In [ ]:

"""Grab the ace tags from claudes memory and serialize them to a file"""
def grab_tags(claude_mem_file, output_file):
    with open (claude_mem_file, 'r', encoding='utf-8') as input_file:
        mem_content = input_file.read()